# Geographic Data Visualization


For today, we're focusing on geographic visualization. Plotly makes it truly, unreasonably easy to create attractive maps.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Creating Basemaps

In [ ]:
coords = pd.DataFrame({
    "lon" : [-118.44300984639733], 
    "lat" : [34.0696449790177],
    "message" : ["Wish you were here"]
})
coords

In [ ]:
fig = px.scatter_mapbox(coords, 
                        lat = "lat",
                        lon = "lon", 
                        hover_name = "message",
                        zoom = 15,
                        height = 300,
                        mapbox_style="open-street-map")

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

The magic happens starting on the third line, when we call px.scatter_mapbox(). The first argument must be a data frame. The lat and lon arguments tell px which columns contain the latitude and longitude coordinates. The hover_name specifies what should appear when we hover over the plotted point with our mouse. zoom controls the initial zoom level of the map, which can subsequently be modified by the user. height allows one to control the aspect ratio. There are many other parameters to px.scatter_mapbox().

The final next two lines control which map tiles are used in the visualization and the amount of whitespace around the visualization. The final line actually displays the map.

Now let's try changing up the zoom level and the map tiles. The positron tiles from CartoDB are very low-contrast, which is very helpful when creating plots that use these tiles as backgrounds.

In [ ]:
# different zoom level, use cartoDB tiles

fig = px.scatter_mapbox(coords,
                        lat = "lat",
                        lon = "lon",
                        hover_name = "message",
                        zoom = 18,
                        height = 300,
                        mapbox_style="carto-positron")

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

# Visualizing Climate Measurement Stations

Let's now use our GHCN data on global temperatures to create some interesting visualizations. As a first step, we'll create a set of markers for different climate stations. First, let's grab the data on stations:

In [ ]:
url = "https://raw.githubusercontent.com/PhilChodrow/PIC16B/master/datasets/noaa-ghcn/station-metadata.csv"
stations = pd.read_csv(url)
stations.head()

For the purposes of geographic plotting, the key columns here are the LATITUDE and LONGITUDE columns. Let's try plotting!

Note that it might take a little while for the map to render. There are 27.5k points, which is kind of a lot!

In [ ]:
fig = px.scatter_mapbox(stations,
                        lat = "LATITUDE",
                        lon = "LONGITUDE",
                        hover_name = "NAME",
                        zoom = 1,
                        height = 300)

fig.update_layout(mapbox_style="open-street-map")
#fig.update_layout(mapbox_style="carto-positron")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

This is cool and interactive, but there are a few shortcomings if we want to display scientific information. It's hard to make comparisons -- for example, it looks like there might be a higher density of stations in the US than in many other areas, but it's hard to be sure from the map above. For comparing densities, heatmaps provided a useful approach. Ploty again makes this unreasonably easy.

In [ ]:
fig = px.density_mapbox(stations, 
                        lat='LATITUDE', 
                        lon='LONGITUDE', 
                        radius=1,
                        zoom=0,
                        height = 300)

fig.update_layout(mapbox_style="carto-positron")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

# Choropleths

A Choropleth Map is a map composed of colored polygons, see [this page](https://www.vividseats.com/blog/most-popular-nfl-teams-by-state-county) for example. It is used to represent spatial variations of a quantity. 
This page documents how to build outline choropleth maps, but you can also build choropleth tile maps using our Mapbox trace types.


## Introduction: main parameters for choropleth outline maps

Making choropleth maps requires two main types of input:

Geometry information:
This can either be a supplied GeoJSON file where each feature has either an id field or some identifying value in properties; or
one of the built-in geometries within plotly: US states and world countries (see below)

A list of values indexed by feature identifier.


### Using built-in geometry information

Plotly choropleth command provides three locationmodes:

- IS0-3: (default choice) first three letters of country name
- USA-states: enables US state geometry. 
- country names:full country name

Mismatching `locations` and `locationmode` does not raise an error, but fails to show a choropleth map.

#### Example 1:

In [ ]:
fig = px.choropleth(locations=["CA", "TX", "NY"], locationmode="USA-states", color=[1,2,3], scope="usa")
fig.show()

#### Gloabl choropleth map

We use two different locationmodes to creat choropleth map.

In [ ]:
df = px.data.gapminder().query("year==2007")
# here, we are using default ISO-3 locationmode!
fig = px.choropleth(df, locations="iso_alpha",
                    color="lifeExp",            # lifeExp is a column of gapminder
                    hover_name="country",       # column to add to hover information
                    color_continuous_scale=px.colors.sequential.Plasma)
fig.show()

In the next example, we use `locationmode = 'country names'`. It gives the same plot.

In [ ]:
df = px.data.gapminder().query("year==2007")
fig = px.choropleth(df, locations="country",
                    locationmode = 'country names',
                    color="lifeExp",            # lifeExp is a column of gapminder
                    hover_name="country",       # column to add to hover information
                    color_continuous_scale=px.colors.sequential.Plasma)
fig.show()

### Using geoJSON file

In this example we set layout.geo.scope to usa to automatically configure the map to display USA-centric data in an appropriate projection.

First, we open json file which is available online

In [ ]:
from urllib.request import urlopen
import json

with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

In [ ]:
counties

Second, we read our datasets which include county fips and unemployment rate

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/fips-unemp-16.csv",
                   dtype={"fips": str})
df

Create choropleth map

In [ ]:
fig = px.choropleth(df, geojson=counties, locations='fips', color='unemp',
                           color_continuous_scale="Viridis",
                           range_color=(0, 12),
                           scope="usa",
                           labels={'unemp':'unemployment rate'}
                          )
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()